# Tune `rf_k_lr`

RF top-K + elastic-net logistic. Repeated stratified CV on the train split; 
writes [`data/processed/tuned/rf_k_lr.json`](../data/processed/tuned/rf_k_lr.json).

**Stage 1 (hyperparameters):** SelectFromModel `max_features` (K), classifier `C`, `l1_ratio`. Select by **max mean PR AUC**.

**Stage 2 (threshold):** sweep `THRESHOLD_GRID` on the same CV folds; select threshold that **minimizes mean BER**.

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if _cwd.name == "tuning" else _cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import scripts.tuning.registry as tuning_registry

importlib.reload(tuning_registry)
from scripts.secom_pipelines import TARGET_COL, feature_columns, load_mart, split_train_test
from scripts.tuning.registry import (
    MODEL_SPECS,
    fit_with_progress,
    run_grid_search,
    save_tuned_params,
    summarize_cv_search,
    tune_classifier_threshold,
    tuned_params_path,
)

MODEL_ID = "rf_k_lr"
spec = MODEL_SPECS[MODEL_ID]


In [2]:
df = load_mart()
feature_cols = feature_columns(df)
train_df, test_df = split_train_test(df)
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].astype(int)
print(len(X_train), "train rows", len(test_df), "test rows (holdout, not used here)")


1253 train rows 314 test rows (holdout, not used here)


In [3]:
param_grid = spec.make_param_grid()
pd.DataFrame([{k: v} for k, v in param_grid.items()])


,preprocess__sensor_mspc__select__max_features,classifier__C,classifier__l1_ratio
0,"[20, 30, 40, 50]",NaN,NaN
1,NaN,"[0.1, 1.0]",NaN
2,NaN,NaN,"[0.5, 0.95]"


In [4]:
search, n_candidates, n_splits, total_fits = run_grid_search(spec, X_train, y_train)
print(f"{MODEL_ID}: {n_candidates} candidates x {n_splits} folds = {total_fits} fits")
search = fit_with_progress(search, X_train, y_train)


rf_k_lr: 16 candidates x 25 folds = 400 fits


GridSearchCV 400 fits:   0%|          | 0/400 [00:00<?, ?it/s]

  0%|          | 0/400 [00:00<?, ?it/s]

Fitting 25 folds for each of 16 candidates, totalling 400 fits


In [5]:
cv_summary, fold_results, aggregated = summarize_cv_search(search, spec)
print("Stage 1 best (mean PR AUC):")
display(aggregated.head(10))


Stage 1 best (mean PR AUC):


,top_k,c,l1_ratio,mean_ber_percent,std_ber_percent,mean_balanced_accuracy,mean_true_positive_percent,std_true_positive_percent,mean_true_negative_percent,std_true_negative_percent,mean_roc_auc,std_roc_auc,mean_pr_auc,std_pr_auc
11,40,1.0,0.95,37.014015,5.977467,0.629860,53.014706,15.171668,72.957265,6.225274,0.690672,0.066413,0.182498,0.053732
10,40,1.0,0.50,37.087733,5.909012,0.629123,52.764706,14.855533,73.059829,6.212305,0.690053,0.066823,0.182112,0.054648
8,40,0.1,0.50,36.573089,6.179363,0.634269,54.426471,14.878574,72.427350,5.799534,0.700307,0.057438,0.181975,0.048721
9,40,0.1,0.95,36.181812,6.061483,0.638182,55.397059,14.719376,72.239316,5.662327,0.703480,0.054536,0.178820,0.046187
7,30,1.0,0.95,38.055744,4.741049,0.619443,50.897059,12.626985,72.991453,5.562783,0.673622,0.064610,0.177291,0.056491
6,30,1.0,0.50,37.896556,4.831296,0.621034,51.147059,12.668552,73.059829,5.577838,0.672800,0.065224,0.176963,0.056818
4,30,0.1,0.50,37.497549,4.789002,0.625025,52.338235,12.117362,72.666667,5.446638,0.683812,0.058266,0.176472,0.049819
5,30,0.1,0.95,37.315548,4.688768,0.626845,53.044118,11.607181,72.324786,5.381603,0.686991,0.055947,0.174966,0.047868
15,50,1.0,0.95,37.078683,6.861469,0.629213,53.705882,18.081036,72.136752,6.701640,0.688008,0.060344,0.172959,0.047872
14,50,1.0,0.50,37.061589,6.839317,0.629384,53.705882,18.081036,72.170940,6.776781,0.687431,0.060949,0.172524,0.048271


In [6]:
threshold_result = tune_classifier_threshold(spec, X_train, y_train, cv_summary)
print(f"Stage 2 best threshold: {threshold_result['best_threshold']:.2f}")
print(f"  mean BER at threshold: {threshold_result['mean_ber_percent']:.2f}%")
display(threshold_result["per_threshold_mean_ber"].head(10))


Threshold CV folds:   0%|          | 0/25 [00:00<?, ?it/s]

Stage 2 best threshold: 0.50
  mean BER at threshold: 37.01%


,threshold,mean_ber_percent
0,0.50,37.014015
1,0.45,38.272373
2,0.55,38.448341
3,0.40,38.583145
4,0.35,39.876634
5,0.60,39.983472
6,0.30,41.209779
7,0.65,41.572398
8,0.25,42.423643
9,0.70,43.511375


In [8]:
payload = save_tuned_params(
    spec,
    cv_summary,
    fold_results,
    aggregated,
    threshold_result=threshold_result,
)
out_path = tuned_params_path(MODEL_ID)
print(f"Wrote {out_path}")
payload["grid_search_best_params"]


Wrote /home/troy/SECOM/data/processed/tuned/rf_k_lr.json


{'preprocess__sensor_mspc__select__max_features': 40,
 'classifier__C': 1.0,
 'classifier__l1_ratio': 0.95}